# Postcards from my Jungle: image generation

Train a small diffusion model **from scratch** on photos of real animals, then sample **50 imagined animals** from it. Based on the course notebook *Train your own diffuser* (W05_02).

**Dataset:** [`Rapidata/Animals-10`](https://huggingface.co/datasets/Rapidata/Animals-10): ~23.5k photos of butterflies, cats, chickens, cows, dogs, elephants, horses, sheep, spiders and squirrels. Mixing very different species is deliberate: the model has no idea what a "species" is, so it blends them into chimeras.

**Before you start:** Runtime → Change runtime type → GPU. Run the cells top to bottom.

## 0. Setup
Fixes a Colab `torchao` / `diffusers` conflict. If the session restarts, that is expected: run the cell again.

In [ ]:
try:
    from diffusers import UNet2DModel
    print("diffusers is ready ✓")
except Exception as e:
    print("Fixing a package conflict, the session will restart…")
    import subprocess, os
    subprocess.run(["pip", "uninstall", "-y", "torchao"])
    os.kill(os.getpid(), 9)

## 1. Settings

Everything I varied between runs lives here. See the experiment log in section 11 for what each change did.

In [ ]:
import torch

# ── Data ────────────────────────────────────────────────────────
HF_DATASET   = "Rapidata/Animals-10"
CLASS_FILTER = None        # e.g. ["Elephant", "Butterfly", "Spider"] to restrict species; None = all ten
MAX_IMAGES   = 10000       # use at most this many images

# ── Training ────────────────────────────────────────────────────
IMAGE_SIZE    = 64         # 64 is the sweet spot on a free T4; 96/128 are far slower
BATCH_SIZE    = 32
TRAIN_STEPS   = 6000      # free T4: ~10k is realistic; on A100/L4 try 20000
LEARNING_RATE = 2e-4
WARMUP_STEPS  = 500
EMA_DECAY     = 0.9995     # slightly smoother than the course default, because the run is longer
PREVIEW_EVERY = 4000
SEED          = 0

# ── Output ──────────────────────────────────────────────────────
NUM_TO_GENERATE = 50       # the assignment asks for 50 imagined animals
SAVE_SIZE       = 256      # saved PNG size (upscaled from IMAGE_SIZE)
OUTPUT_DIR      = "images"
MODEL_FILE      = "jungle_ema.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = device.type == "cuda" and torch.cuda.get_device_capability()[0] >= 8   # real bf16: A100/L4 and newer
USE_FP16 = device.type == "cuda" and not USE_BF16                                # T4: use fp16
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
AMP_ON = device.type == "cuda"
torch.manual_seed(SEED)
print("Device:", torch.cuda.get_device_name() if device.type == "cuda" else "CPU (far too slow, switch to GPU)")

## 2. Load the real animals

In [ ]:
from datasets import load_dataset, Image as HFImage

dataset = load_dataset(HF_DATASET, split="train")
image_col = next(n for n, f in dataset.features.items() if isinstance(f, HFImage))

if CLASS_FILTER:
    names = dataset.features["label"].names
    keep = {names.index(c) for c in CLASS_FILTER}
    dataset = dataset.filter(lambda ex: ex["label"] in keep)

dataset = dataset.shuffle(seed=SEED).select(range(min(MAX_IMAGES, len(dataset))))
print(f"Using {len(dataset)} images (column '{image_col}')")

## 3. Prepare the images
Centre-crop to a square, shrink to `IMAGE_SIZE`, random horizontal flip, scale to [-1, 1].

In [ ]:
from torchvision import transforms
from diffusers.utils import make_image_grid
from PIL import Image
from tqdm.auto import tqdm

def make_preprocess(flip=True):
    steps = [transforms.Resize(IMAGE_SIZE), transforms.CenterCrop(IMAGE_SIZE)]
    if flip: steps.append(transforms.RandomHorizontalFlip())
    steps += [transforms.ToTensor(), transforms.Normalize([0.5], [0.5])]
    return transforms.Compose(steps)

def to_pil(batch, size=128, resample=Image.NEAREST):
    batch = ((batch.clamp(-1, 1) + 1) / 2 * 255).byte().cpu()
    return [Image.fromarray(i.permute(1, 2, 0).numpy()).resize((size, size), resample) for i in batch]

# Resize every image ONCE and keep the whole dataset on the GPU as uint8.
# (Decoding full-size JPEGs on Colab's 2 CPU cores every step was the bottleneck.)
small = make_preprocess(flip=False)
data = torch.stack([
    ((small(dataset[i][image_col].convert("RGB")) + 1) * 127.5).round().byte()
    for i in tqdm(range(len(dataset)), desc="Resizing")
]).to(device)
print("Dataset on GPU:", tuple(data.shape), f"{data.numel() / 1e6:.0f} MB")

def get_batch(bs):
    idx = torch.randint(0, data.shape[0], (bs,), device=device)
    x = data[idx].float() / 127.5 - 1
    flip = torch.rand(bs, device=device) < 0.5                      # random horizontal flip
    return torch.where(flip[:, None, None, None], x.flip(3), x)

batch_size = min(BATCH_SIZE, len(data))
print(f"{TRAIN_STEPS} steps × batch {batch_size} ≈ {TRAIN_STEPS * batch_size / len(data):.1f} passes over the data")
make_image_grid(to_pil(get_batch(8)), rows=1, cols=8)

## 4. Model, noise schedule, optimizer

In [ ]:
from diffusers import UNet2DModel, DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_cosine_schedule_with_warmup

def make_model():
    return UNet2DModel(
        sample_size=IMAGE_SIZE, in_channels=3, out_channels=3, layers_per_block=2,
        block_out_channels=(64, 128, 256, 512),
        down_block_types=("DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
        up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D"),
    ).to(device)

model = make_model()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
noise_scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule="squaredcos_cap_v2")
ema = EMAModel(model.parameters(), decay=EMA_DECAY); ema.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
lr_scheduler = get_cosine_schedule_with_warmup(optimizer, min(WARMUP_STEPS, TRAIN_STEPS // 10), TRAIN_STEPS)

## 5. Sampling function (DDIM)

In [ ]:
from diffusers import DDIMScheduler

@torch.no_grad()
def generate(unet, n=8, steps=100, seed=0):
    sampler = DDIMScheduler.from_config(noise_scheduler.config)
    sampler.set_timesteps(steps)
    g = torch.Generator(device=device).manual_seed(seed)
    x = torch.randn(n, 3, IMAGE_SIZE, IMAGE_SIZE, generator=g, device=device)
    was_training = unet.training; unet.eval()
    for t in sampler.timesteps:
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_ON):
            pred = unet(x, t).sample
        x = sampler.step(pred.float(), t, x).prev_sample
    unet.train(was_training)
    return x.clamp(-1, 1)

def preview_with_ema(n=8, steps=50):
    ema.store(model.parameters()); ema.copy_to(model.parameters())
    imgs = to_pil(generate(model, n=n, steps=steps))
    ema.restore(model.parameters())
    return make_image_grid(imgs, rows=1, cols=n)

## 6. Train
With the data on the GPU this should run at several steps per second on a T4 (check the progress bar after a minute: if it shows more than ~0.5 s/it, lower `BATCH_SIZE` or `TRAIN_STEPS`). Previews appear every `PREVIEW_EVERY` steps.

In [ ]:
import time
!nvidia-smi --query-gpu=name,utilization.gpu,memory.used,clocks.sm,clocks.max.sm,temperature.gpu --format=csv

def make_light():
    return UNet2DModel(
        sample_size=IMAGE_SIZE, in_channels=3, out_channels=3, layers_per_block=1,
        block_out_channels=(64, 128, 256),
        down_block_types=("DownBlock2D", "DownBlock2D", "AttnDownBlock2D"),
        up_block_types=("AttnUpBlock2D", "UpBlock2D", "UpBlock2D"),
    ).to(device)

def bench(m, n=10):
    x = torch.randn(32, 3, 64, 64, device=device); t = torch.randint(0, 1000, (32,), device=device)
    opt = torch.optim.AdamW(m.parameters())
    for i in range(n + 3):
        if i == 3: torch.cuda.synchronize(); t0 = time.time()
        with torch.autocast("cuda", dtype=torch.float16):
            loss = m(x, t).sample.float().pow(2).mean()
        loss.backward(); opt.step(); opt.zero_grad()
    torch.cuda.synchronize()
    return (time.time() - t0) / n

print("current model:", round(bench(make_model()), 2), "s/step")
print("light model:  ", round(bench(make_light()), 2), "s/step")

In [ ]:
import time
def sync(): torch.cuda.synchronize(); return time.time()
model.train(); T = {}
def tick(k, t0): t1 = sync(); T[k] = T.get(k, 0) + (t1 - t0); return t1

for s in range(13):
    if s == 3: T.clear()          # skip warm-up steps
    t0 = sync()
    clean = get_batch(batch_size);                         t0 = tick("get_batch", t0)
    noise = torch.randn_like(clean)
    t = torch.randint(0, 1000, (batch_size,), device=device).long()
    noisy = noise_scheduler.add_noise(clean, noise, t);    t0 = tick("add_noise", t0)
    with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_ON):
        pred = model(noisy, t).sample
    loss = F.mse_loss(pred.float(), noise);                t0 = tick("forward", t0)
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward();                         t0 = tick("backward", t0)
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); t0 = tick("unscale+clip", t0)
    scaler.step(optimizer); scaler.update();               t0 = tick("optimizer", t0)
    lr_scheduler.step();                                   t0 = tick("lr_sched", t0)
    ema.step(model.parameters());                          t0 = tick("ema", t0)
    losses.append(loss.item());                            t0 = tick("item", t0)

for k, v in T.items(): print(f"{k:<13}{v / 10:.3f} s/step")

In [ ]:
import torch.nn.functional as F
from tqdm.auto import tqdm
from IPython.display import display

scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)   # only needed for fp16 (T4)
losses = []
model.train()
progress = tqdm(range(1, TRAIN_STEPS + 1), desc="Training")
for step in progress:
    clean = get_batch(batch_size)
    noise = torch.randn_like(clean)
    t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (batch_size,), device=device).long()
    noisy = noise_scheduler.add_noise(clean, noise, t)
    with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_ON):
        pred = model(noisy, t).sample
    loss = F.mse_loss(pred.float(), noise)
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer); scaler.update()
    lr_scheduler.step(); ema.step(model.parameters())
    losses.append(loss.item())
    if step % 50 == 0: progress.set_postfix(loss=f"{sum(losses[-50:]) / 50:.4f}")
    if step % PREVIEW_EVERY == 0 or step == TRAIN_STEPS:
        progress.write(f"Step {step}: loss {sum(losses[-100:]) / len(losses[-100:]):.4f}")
        display(preview_with_ema())

## 7. Training loss

In [ ]:
from matplotlib import pyplot as plt
def smooth(v, k=100): return [sum(v[max(0, i-k):i+1]) / len(v[max(0, i-k):i+1]) for i in range(len(v))]
plt.figure(figsize=(10, 4)); plt.plot(losses, alpha=.3); plt.plot(smooth(losses))
plt.xlabel("step"); plt.ylabel("loss"); plt.title("Training loss"); plt.show()

## 8. Save the EMA weights
Uncomment the Drive lines to keep the model after the session ends.

In [ ]:
ema.copy_to(model.parameters())
torch.save(model.state_dict(), MODEL_FILE)
print("Saved", MODEL_FILE)
# from google.colab import drive; drive.mount("/content/drive")
# torch.save(model.state_dict(), f"/content/drive/MyDrive/{MODEL_FILE}")

## 9. Generate the 50 imagined animals

Saves `images/animal_00.png` … `animal_49.png` (upscaled with Lanczos, which is why they look soft, a lo-fi feature rather than a bug) and a contact sheet, then downloads a zip. **Put the PNGs in the `images/` folder of the repo.**

In [ ]:
import os, shutil
trained = make_model()
trained.load_state_dict(torch.load(MODEL_FILE, map_location=device, weights_only=True))

generated = generate(trained, n=NUM_TO_GENERATE, steps=100, seed=42)
images = to_pil(generated, size=SAVE_SIZE, resample=Image.LANCZOS)

os.makedirs(OUTPUT_DIR, exist_ok=True)
for i, img in enumerate(images):
    img.save(f"{OUTPUT_DIR}/animal_{i:02d}.png")

cols = 10; rows = -(-len(images) // cols)
small = [im.resize((128, 128)) for im in images]
small += [Image.new("RGB", (128, 128), "white")] * (rows * cols - len(small))
grid = make_image_grid(small, rows=rows, cols=cols)
grid.save("contact_sheet.png")

zip_path = shutil.make_archive("images", "zip", OUTPUT_DIR)
try:
    from google.colab import files; files.download(zip_path)
except ImportError:
    print("Saved", zip_path)
grid

## 10. Inventing or copying?
Each generated image (top) next to its nearest training image (bottom). Near-identical pairs mean the model memorised: use more data or fewer steps.

In [ ]:
check = make_preprocess(flip=False)
n_check = min(len(dataset), 2000)
train_t = torch.stack([check(dataset[i][image_col].convert("RGB")) for i in range(n_check)]).to(device)
n_show = 8
nearest = torch.cdist(generated[:n_show].flatten(1), train_t.flatten(1)).argmin(1)
pairs = []
for g, idx in zip(to_pil(generated[:n_show]), nearest.tolist()):
    pairs += [g, to_pil(train_t[idx:idx+1])[0]]
make_image_grid(pairs[0::2] + pairs[1::2], rows=2, cols=n_show)

## 11. Experiment log

Fill this in with *your* own runs (I changed one setting at a time). The first row is the configuration in section 1; the others are the variations worth trying.

| Run | Images | Size | Steps | What I saw |
|---|---|---|---|---|
| A (final) | 10,000 | 64 | 20,000 | *(write what you saw)* |
| B | 1,000 | 64 | 20,000 | *(does it copy? check section 10)* |
| C | 10,000 | 32 | 5,000 | *(fast, but what is still recognisable?)* |
| D | 10,000 | 64 | 5,000 | *(blobs vs. animals)* |
| E | 3 species only (`CLASS_FILTER`) | 64 | 20,000 | *(more coherent, fewer chimeras?)* |